# CNN-EXP-034 — Dual-path RF2053 + local motif branch + per-base Poisson

Это первый **production-oriented** эксперимент после анализа `HARD_FP` и `LOW_TP`.

## Что именно проверяем

Наши error-analysis эксперименты показали две вещи:

```text
1. LOW-score TP и HIGH-score TP различаются по local k-mer vocabulary
   даже после строгого matching по signal strength.

2. ISOLATED hard FP локально очень похожи на настоящие TP,
   поэтому одной короткой motif-логики недостаточно.
```

Отсюда архитектура:

```text
                         ┌─ LOCAL BRANCH ───────────────┐
raw DNA 5ch ────────────┤  RF ≈ 35 bp                 │
                         │  core motif vocabulary       │
                         │                              ├─ concat
                         └─ CONTEXT BRANCH ─────────────┘    ↓
                            RF = 2053 bp                  fusion
                            старый сильный backbone          ↓
                                                        per-base
                                                        log-rate
```

## Почему пока НЕ добавляем hard mining

Сначала нужен чистый тест новой постановки:

```text
dual-path representation
+
per-base count objective
```

Если сразу одновременно добавить hard mining, отдельный sampler и новые веса loss,
мы не поймём, откуда пришёл прирост.

Hard-positive / hard-negative curriculum — следующий эксперимент, если EXP034
покажет перспективу.

## Objective

Финальный output — **один log-rate на каждую bp**.

Обучение:

```text
Poisson NLL(count | exp(log_rate))
```

То есть модель сразу учится:

- отличать `count=0` от `count>0` → полезно для AP;
- ранжировать силу положительных TSS → полезно для EPIC Spearman.

## Важное отличие от EXP033

EXP033 решал:

```text
"есть ли TSS где-нибудь внутри 32 bp?"
```

EXP034 решает:

```text
"какой initiation rate у каждой конкретной bp?"
```

32-bp метрика остаётся только как **diagnostic bridge**, чтобы можно было
сравнить результат с EXP033.


In [1]:
from pathlib import Path
import copy
import gc
import json
import math
import sys
import time
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_directml

from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (
        CURRENT_DIR,
        *CURRENT_DIR.parents,
    )
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_DIR),
    )

from ml_project.epic_baseline import (
    load_baseline_sequences,
)

from ml_project.epic_cnn import (
    DirectMLAdam,
    ProfileWindowConfig,
    ProfileWindowGenerator,
    ResidualBlock,
    create_cnn_presence_intensity,
)

from ml_project.epic_data import (
    load_split,
)

print("Project:", PROJECT_ROOT)


Project: D:\Projects\EPIC\EPIC


In [2]:
device = torch_directml.device()

print(
    "DirectML device:",
    device,
)

print(
    "GPU:",
    torch_directml.device_name(0),
)


DirectML device: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


In [3]:
# ============================================================
# DATA / WINDOW CONFIG
# ============================================================

split = load_split(
    PROJECT_ROOT
)

sequences, fasta_alphabet = (
    load_baseline_sequences(
        split
    )
)

train_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "train"
            )[
                "contig"
            ].astype(str)
        )
    )
)

validation_contigs = tuple(
    sorted(
        set(
            split.intervals(
                "validation"
            )[
                "contig"
            ].astype(str)
        )
    )
)

assert set(
    train_contigs
).isdisjoint(
    validation_contigs
)

TARGET_LENGTH = 2048

# RF2053 has radius ~1026 bp.
# This gives every target bp enough real sequence context on both sides
# instead of arbitrary zero-padding at 32-bp tile boundaries.
CONTEXT_FLANK = 1026

INPUT_LENGTH = (
    TARGET_LENGTH
    + 2 * CONTEXT_FLANK
)

REGION_WIDTH = 32

BLOCK_SETTINGS_RF2053 = (
    (7, 1),
    (3, 2),
    (3, 4),
    (3, 8),
    (3, 16),
    (3, 32),
    (3, 64),
    (3, 128),
    (3, 256),
)

CONTEXT_RF_BP = (
    1
    + sum(
        2
        * (
            kernel_size
            - 1
        )
        * dilation
        for kernel_size, dilation
        in BLOCK_SETTINGS_RF2053
    )
)

assert CONTEXT_RF_BP == 2053
assert TARGET_LENGTH % REGION_WIDTH == 0

profile_config = (
    ProfileWindowConfig(
        target_length=TARGET_LENGTH,
        context_flank=CONTEXT_FLANK,
        positive_branch_fraction=0.5,
    )
)

train_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=42,
    )
)

validation_generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="validation",
        config=profile_config,
        seed=42,
    )
)

display(
    split.summary
)

print(
    "Train contigs:",
    len(
        train_contigs
    ),
)

print(
    "Validation contigs:",
    len(
        validation_contigs
    ),
)

print(
    "Input length:",
    INPUT_LENGTH,
)

print(
    "Central target:",
    TARGET_LENGTH,
)

print(
    "Context flank:",
    CONTEXT_FLANK,
)

print(
    "Context branch theoretical RF:",
    CONTEXT_RF_BP,
)


,split,contigs,intervals_both_strands,whitelist_bp,position_strand_count,percent_of_all_positions,positive_positions,negative_positions,positive_percent
0,train,9,168998,114032080,228064160,59.316784,170443.0,227893717.0,0.074735
1,validation,3,60386,41903743,83807486,21.797334,56205.0,83751281.0,0.067064
2,test,3,54846,36306695,72613390,18.885882,NaN,NaN,NaN


Train contigs: 9
Validation contigs: 3
Input length: 4100
Central target: 2048
Context flank: 1026
Context branch theoretical RF: 2053


In [4]:
# ============================================================
# REAL BATCH CHECK
# ============================================================

smoke_batch = (
    train_generator
    .sample_batch(
        batch_size=2
    )
)

print(
    "x:",
    tuple(
        smoke_batch.x.shape
    ),
)

print(
    "x_both_strands:",
    tuple(
        smoke_batch
        .x_both_strands
        .shape
    ),
)

print(
    "count:",
    tuple(
        smoke_batch
        .count
        .shape
    ),
)

print(
    "mask:",
    tuple(
        smoke_batch
        .mask
        .shape
    ),
)

print(
    "plus loss weight:",
    tuple(
        smoke_batch
        .plus_loss_weight
        .shape
    ),
)

print(
    "minus reverse loss weight:",
    tuple(
        smoke_batch
        .minus_loss_weight_reverse
        .shape
    ),
)

assert (
    smoke_batch
    .x_both_strands
    .shape[-1]
    == INPUT_LENGTH
)

assert (
    smoke_batch
    .x_both_strands
    .shape[1]
    == 5
)

assert (
    smoke_batch
    .count
    .shape[-1]
    == TARGET_LENGTH
)


x: (2, 5, 4100)
x_both_strands: (4, 5, 4100)
count: (2, 2, 2048)
mask: (2, 2, 2048)
plus loss weight: (2, 1, 2048)
minus reverse loss weight: (2, 1, 2048)


## Архитектура

Здесь намеренно сохраняем сильную часть EXP033:

```text
context backbone:
RF = 2053 bp
32 channels
base resolution
```

И добавляем отдельный локальный путь:

```text
Conv11
→ ResBlock(k5,d1)
→ ResBlock(k5,d2)

local RF = 35 bp
```

Почему именно такой local branch:

- наши HIGH/LOW TP probes показали сильный `k-mer` сигнал;
- branch никогда не downsample'ится;
- 35 bp достаточно широк, чтобы видеть комбинации коротких motifs,
  но он не смешивает их с килобазным контекстом.

После этого local и context features **конкатенируются**, а не усредняются.
Модель сама решает, какой путь важнее для каждой позиции.


In [5]:
# ============================================================
# DUAL-PATH PER-BASE MODEL
# ============================================================


class LocalMotifBranch(
    nn.Module
):
    def __init__(
        self,
        input_channels=5,
        channels=32,
    ):
        super().__init__()

        self.stem = (
            nn.Sequential(
                nn.Conv1d(
                    input_channels,
                    channels,
                    kernel_size=11,
                    padding=5,
                ),
                nn.BatchNorm1d(
                    channels
                ),
                nn.GELU(),
            )
        )

        self.block1 = (
            ResidualBlock(
                channels=channels,
                kernel_size=5,
                dilation=1,
            )
        )

        self.block2 = (
            ResidualBlock(
                channels=channels,
                kernel_size=5,
                dilation=2,
            )
        )

    def forward(
        self,
        x,
    ):
        x = self.stem(
            x
        )

        x = self.block1(
            x
        )

        x = self.block2(
            x
        )

        return x


class DualPathInitiationModel(
    nn.Module
):
    def __init__(
        self,
        input_channels=5,
        context_channels=32,
        local_channels=32,
        fusion_channels=48,
    ):
        super().__init__()

        context_model = (
            create_cnn_presence_intensity(
                input_channels=input_channels,
                channels=context_channels,
                block_settings=BLOCK_SETTINGS_RF2053,
            )
        )

        # Reuse only the proven RF2053 feature extractor.
        self.context_backbone = (
            context_model.backbone
        )

        self.local_branch = (
            LocalMotifBranch(
                input_channels=input_channels,
                channels=local_channels,
            )
        )

        self.fusion = (
            nn.Sequential(
                nn.Conv1d(
                    context_channels
                    + local_channels,
                    fusion_channels,
                    kernel_size=1,
                ),
                nn.GELU(),
                nn.Conv1d(
                    fusion_channels,
                    fusion_channels,
                    kernel_size=1,
                ),
                nn.GELU(),
            )
        )

        self.rate_head = (
            nn.Conv1d(
                fusion_channels,
                1,
                kernel_size=1,
            )
        )

        # Sparse initiation prior.
        nn.init.constant_(
            self.rate_head.bias,
            -8.0,
        )

    def forward(
        self,
        x,
    ):
        context_features = (
            self.context_backbone(
                x
            )
        )

        local_features = (
            self.local_branch(
                x
            )
        )

        if (
            context_features.shape[-1]
            != local_features.shape[-1]
        ):
            raise ValueError(
                (
                    context_features.shape,
                    local_features.shape,
                )
            )

        fused = torch.cat(
            (
                context_features,
                local_features,
            ),
            dim=1,
        ).contiguous()

        fused = self.fusion(
            fused
        )

        return self.rate_head(
            fused
        )


def create_model():
    return (
        DualPathInitiationModel()
    )


model_cpu = (
    create_model()
)

trainable_params = sum(
    p.numel()
    for p in (
        model_cpu
        .parameters()
    )
    if p.requires_grad
)

print(
    "Trainable params:",
    f"{trainable_params:,}",
)

print(
    "Context RF:",
    CONTEXT_RF_BP,
)

print(
    "Local RF:",
    35,
)

del model_cpu


Trainable params: 93,649
Context RF: 2053
Local RF: 35


In [6]:
# ============================================================
# FORWARD / COUNT TARGETS / POISSON LOSS
# ============================================================


def crop_target(
    x,
):
    return (
        x[
            :,
            :,
            CONTEXT_FLANK:
            CONTEXT_FLANK
            + TARGET_LENGTH
        ]
        .contiguous()
    )


def forward_both_oriented(
    model,
    x_both,
    batch_size,
):
    all_log_rate = (
        model(
            x_both
        )
    )

    if (
        all_log_rate.shape[-1]
        != INPUT_LENGTH
    ):
        raise ValueError(
            (
                all_log_rate.shape,
                INPUT_LENGTH,
            )
        )

    plus = crop_target(
        all_log_rate[
            :batch_size
        ]
    )

    minus_reverse = crop_target(
        all_log_rate[
            batch_size:
        ]
    )

    return (
        plus,
        minus_reverse,
    )


def batch_count_targets(
    batch,
):
    # batch.count is genomic orientation [B, 2, TARGET_LENGTH].
    plus_count = (
        batch.count[
            :,
            0:
            1,
            :,
        ]
        .float()
        .contiguous()
    )

    minus_count_reverse = (
        torch.flip(
            batch.count[
                :,
                1:
                2,
                :,
            ],
            dims=[
                -1
            ],
        )
        .float()
        .contiguous()
    )

    return (
        plus_count,
        minus_count_reverse,
    )


def masked_poisson_nll(
    log_rate,
    count,
    valid,
):
    # Explicit float32 path for numerical stability.
    z = (
        log_rate
        .float()
        .clamp(
            min=-20.0,
            max=15.0,
        )
    )

    y = (
        count
        .float()
    )

    m = (
        valid
        .float()
    )

    # Constant lgamma(y+1) is omitted:
    # it does not affect gradients or model comparison.
    element = (
        torch.exp(
            z
        )
        - y * z
    )

    denom = torch.clamp(
        m.sum(),
        min=1.0,
    )

    return (
        element
        * m
    ).sum() / denom


In [7]:
# ============================================================
# ONE REAL DIRECTML BACKWARD STEP
# ============================================================

torch.manual_seed(
    123
)

smoke_model = (
    create_model()
    .to(device)
)

smoke_optimizer = (
    DirectMLAdam(
        smoke_model.parameters(),
        lr=1e-3,
    )
)

smoke_model.train()

B = 2

smoke_batch = (
    train_generator
    .sample_batch(
        batch_size=B
    )
)

x_gpu = (
    smoke_batch
    .x_both_strands
    .to(device)
)

(
    plus_count,
    minus_count_reverse,
) = batch_count_targets(
    smoke_batch
)

plus_valid = (
    smoke_batch
    .plus_loss_weight
    .float()
)

minus_valid_reverse = (
    smoke_batch
    .minus_loss_weight_reverse
    .float()
)

smoke_optimizer.zero_grad(
    set_to_none=True
)

with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    (
        plus_log_rate,
        minus_log_rate_reverse,
    ) = forward_both_oriented(
        smoke_model,
        x_gpu,
        batch_size=B,
    )

    plus_loss = (
        masked_poisson_nll(
            plus_log_rate,
            plus_count.to(
                device
            ),
            plus_valid.to(
                device
            ),
        )
    )

    minus_loss = (
        masked_poisson_nll(
            minus_log_rate_reverse,
            minus_count_reverse.to(
                device
            ),
            minus_valid_reverse.to(
                device
            ),
        )
    )

    smoke_loss = (
        plus_loss
        + minus_loss
    ) / 2.0

    smoke_loss.backward()
    smoke_optimizer.step()

fallback = [
    str(
        w.message
    )
    for w in caught
    if "fall back"
    in str(
        w.message
    ).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(
            fallback
        )
    )

print(
    "plus:",
    tuple(
        plus_log_rate.shape
    ),
)

print(
    "minus reverse:",
    tuple(
        minus_log_rate_reverse.shape
    ),
)

print(
    "Poisson smoke loss:",
    float(
        smoke_loss
        .detach()
        .cpu()
        .item()
    ),
)

print(
    "DirectML backward: OK"
)

del (
    smoke_model,
    smoke_optimizer,
    smoke_batch,
    x_gpu,
    plus_log_rate,
    minus_log_rate_reverse,
    smoke_loss,
)

gc.collect()


plus: (2, 1, 2048)
minus reverse: (2, 1, 2048)
Poisson smoke loss: 2.695136070251465
DirectML backward: OK


57

In [8]:
# ============================================================
# TRAIN — 10K ARCHITECTURE SCREEN
# ============================================================

STEPS = 10_000
SAVE_EVERY = 2_500
BATCH_SIZE = 4
LOG_EVERY = 100
SEED = 42
LEARNING_RATE = 1e-3

EXPERIMENT = (
    "CNN-EXP-034"
)

ARCHITECTURE = (
    "dualpath_local35_context2053_perbase_poisson"
)

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP034_DUALPATH_RF2053_PERBASE_POISSON"
    / "run_001"
)

RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HISTORY_PATH = (
    RUN_DIR
    / "training_history.csv"
)

torch.manual_seed(
    SEED
)

generator = (
    ProfileWindowGenerator(
        prepared_split=split,
        sequences=sequences,
        part="train",
        config=profile_config,
        seed=SEED,
    )
)

model = (
    create_model()
    .to(device)
)

optimizer = (
    DirectMLAdam(
        model.parameters(),
        lr=LEARNING_RATE,
    )
)

model.train()

history = []

ema_loss = None
seen_target_positions = 0
started = time.perf_counter()


def tree_to_cpu(
    value
):
    if torch.is_tensor(
        value
    ):
        return (
            value.detach()
            .cpu()
            .clone()
        )

    if isinstance(
        value,
        dict,
    ):
        return {
            key: tree_to_cpu(
                item
            )
            for key, item
            in value.items()
        }

    if isinstance(
        value,
        list,
    ):
        return [
            tree_to_cpu(
                item
            )
            for item in value
        ]

    if isinstance(
        value,
        tuple,
    ):
        return tuple(
            tree_to_cpu(
                item
            )
            for item in value
        )

    return value


def checkpoint_path(
    step
):
    return (
        RUN_DIR
        / (
            f"checkpoint_step_"
            f"{step:06d}.pt"
        )
    )


def save_checkpoint(
    step
):
    path = checkpoint_path(
        step
    )

    checkpoint = {
        "experiment": (
            EXPERIMENT
        ),
        "architecture": (
            ARCHITECTURE
        ),
        "step": int(
            step
        ),
        "model_state_dict": {
            k: (
                v.detach()
                .cpu()
                .clone()
            )
            for k, v
            in model
            .state_dict()
            .items()
        },
        "optimizer_state_dict": (
            tree_to_cpu(
                optimizer
                .state_dict()
            )
        ),
        "generator_rng_state": (
            copy.deepcopy(
                generator
                .rng
                .bit_generator
                .state
            )
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "target_length": int(
            TARGET_LENGTH
        ),
        "context_flank": int(
            CONTEXT_FLANK
        ),
        "input_length": int(
            INPUT_LENGTH
        ),
        "context_rf_bp": int(
            CONTEXT_RF_BP
        ),
        "local_rf_bp": 35,
        "block_settings": (
            BLOCK_SETTINGS_RF2053
        ),
        "objective": (
            "masked_per_base_poisson_log_rate"
        ),
        "batch_size": int(
            BATCH_SIZE
        ),
        "learning_rate": float(
            LEARNING_RATE
        ),
        "seed": int(
            SEED
        ),
        "training_budget": int(
            STEPS
        ),
        "ema_loss": float(
            ema_loss
        ),
        "elapsed_minutes": float(
            (
                time.perf_counter()
                - started
            )
            / 60.0
        ),
    }

    torch.save(
        checkpoint,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )

    print()
    print(
        f"=== CHECKPOINT SAVED @ {step:,} ==="
    )

    print(
        "EMA loss:",
        round(
            float(
                ema_loss
            ),
            6,
        ),
    )

    print(
        "Path:",
        path,
    )

    print()


with warnings.catch_warnings(
    record=True
) as caught:

    warnings.simplefilter(
        "always"
    )

    for step in range(
        1,
        STEPS + 1,
    ):
        batch = (
            generator
            .sample_batch(
                batch_size=BATCH_SIZE
            )
        )

        x_gpu = (
            batch
            .x_both_strands
            .to(device)
        )

        (
            plus_count,
            minus_count_reverse,
        ) = batch_count_targets(
            batch
        )

        plus_valid = (
            batch
            .plus_loss_weight
            .float()
        )

        minus_valid_reverse = (
            batch
            .minus_loss_weight_reverse
            .float()
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        (
            plus_log_rate,
            minus_log_rate_reverse,
        ) = forward_both_oriented(
            model,
            x_gpu,
            batch_size=BATCH_SIZE,
        )

        plus_loss = (
            masked_poisson_nll(
                plus_log_rate,
                plus_count.to(
                    device
                ),
                plus_valid.to(
                    device
                ),
            )
        )

        minus_loss = (
            masked_poisson_nll(
                minus_log_rate_reverse,
                minus_count_reverse.to(
                    device
                ),
                minus_valid_reverse.to(
                    device
                ),
            )
        )

        total_loss = (
            plus_loss
            + minus_loss
        ) / 2.0

        total_value = float(
            total_loss
            .detach()
            .cpu()
            .item()
        )

        if not math.isfinite(
            total_value
        ):
            raise FloatingPointError(
                f"Non-finite loss at step {step}"
            )

        total_loss.backward()
        optimizer.step()

        if ema_loss is None:
            ema_loss = (
                total_value
            )
        else:
            ema_loss = (
                0.95
                * ema_loss
                + 0.05
                * total_value
            )

        seen_target_positions += int(
            batch.mask.sum()
        )

        elapsed = (
            time.perf_counter()
            - started
        )

        row = {
            "step": int(
                step
            ),
            "loss": total_value,
            "plus_loss": float(
                plus_loss
                .detach()
                .cpu()
                .item()
            ),
            "minus_loss": float(
                minus_loss
                .detach()
                .cpu()
                .item()
            ),
            "ema_loss": float(
                ema_loss
            ),
            "target_positions_per_second": float(
                seen_target_positions
                / elapsed
            ),
        }

        history.append(
            row
        )

        if (
            step == 1
            or step
            % LOG_EVERY
            == 0
        ):
            print(
                f"step={step:6d} "
                f"loss={total_value:.5f} "
                f"ema={ema_loss:.5f} "
                f"target-pos/s="
                f"{seen_target_positions / elapsed:,.0f}"
            )

        if (
            step % SAVE_EVERY == 0
            or step == STEPS
        ):
            save_checkpoint(
                step
            )

fallback = [
    str(
        w.message
    )
    for w in caught
    if "fall back"
    in str(
        w.message
    ).lower()
]

if fallback:
    raise RuntimeError(
        "DirectML fallback: "
        + " | ".join(
            fallback
        )
    )

pd.DataFrame(
    history
).to_csv(
    HISTORY_PATH,
    index=False,
)

print()
print(
    "Elapsed min:",
    round(
        (
            time.perf_counter()
            - started
        )
        / 60.0,
        2,
    ),
)

print(
    "Final EMA loss:",
    ema_loss,
)

print(
    "Final checkpoint:",
    checkpoint_path(
        STEPS
    ),
)


step=     1 loss=12.03773 ema=12.03773 target-pos/s=9,544
step=   100 loss=-16.48337 ema=-1.08922 target-pos/s=38,729
step=   200 loss=-1.94794 ema=-1.82484 target-pos/s=39,645
step=   300 loss=1.01928 ema=13.07992 target-pos/s=39,875
step=   400 loss=1.31709 ema=5.44045 target-pos/s=40,195
step=   500 loss=2.65065 ema=-0.62152 target-pos/s=40,576
step=   600 loss=3.50382 ema=-1.36744 target-pos/s=40,691
step=   700 loss=1.33709 ema=-1.72304 target-pos/s=40,878
step=   800 loss=-1.06510 ema=-1.85533 target-pos/s=41,064
step=   900 loss=3.35111 ema=-4.01285 target-pos/s=40,997
step=  1000 loss=0.22224 ema=-0.97785 target-pos/s=40,933
step=  1100 loss=2.38134 ema=-17.39918 target-pos/s=41,015
step=  1200 loss=1.02941 ema=-0.70116 target-pos/s=41,055
step=  1300 loss=-0.12816 ema=-2.33099 target-pos/s=41,088
step=  1400 loss=3.37999 ema=-5.22683 target-pos/s=41,129
step=  1500 loss=1.59669 ema=-1.00949 target-pos/s=41,232
step=  1600 loss=0.65846 ema=-0.71139 target-pos/s=41,509
step=  17

In [9]:
# ============================================================
# FULL HOLDOUT TILE INDEX
# ============================================================

validation_tiles = (
    validation_generator
    .tile_index
    .copy()
)

print(
    "Tiles:",
    len(
        validation_tiles
    ),
)

print(
    "Position-strand:",
    int(
        validation_tiles[
            "allowed_position_strand"
        ].sum()
    ),
)

print(
    "Positives:",
    int(
        validation_tiles[
            "positive_positions"
        ].sum()
    ),
)

print(
    "Contigs:",
    validation_tiles[
        "contig"
    ].nunique(),
)


Tiles: 24700
Position-strand: 83807486
Positives: 56205
Contigs: 3


In [2]:
# ============================================================
# FULL HOLDOUT EVALUATION HELPERS
# ============================================================


def evaluate_full_holdout(
    model,
    tiles,
    *,
    batch_size=4,
    progress_every=250,
):
    was_training = (
        model.training
    )

    model.eval()

    score_chunks = []
    count_chunks = []

    regional_score_chunks = []
    regional_positive_chunks = []
    regional_tss_count_chunks = []

    started = (
        time.perf_counter()
    )

    positions = 0

    try:
        with warnings.catch_warnings(
            record=True
        ) as caught:

            warnings.simplefilter(
                "always"
            )

            with torch.no_grad():
                for batch_no, start in enumerate(
                    range(
                        0,
                        len(
                            tiles
                        ),
                        batch_size,
                    ),
                    1,
                ):
                    selected = (
                        tiles.iloc[
                            start:
                            start
                            + batch_size
                        ]
                    )

                    batch = (
                        validation_generator
                        .make_batch(
                            selected
                        )
                    )

                    B = len(
                        selected
                    )

                    x_gpu = (
                        batch
                        .x_both_strands
                        .to(device)
                    )

                    (
                        plus_log_rate,
                        minus_log_rate_reverse,
                    ) = (
                        forward_both_oriented(
                            model,
                            x_gpu,
                            batch_size=B,
                        )
                    )

                    plus_score = (
                        plus_log_rate[
                            :,
                            0,
                            :,
                        ]
                        .detach()
                        .cpu()
                        .numpy()
                    )

                    minus_score = (
                        torch.flip(
                            minus_log_rate_reverse,
                            dims=[
                                -1
                            ],
                        )[
                            :,
                            0,
                            :,
                        ]
                        .detach()
                        .cpu()
                        .numpy()
                    )

                    score = np.stack(
                        (
                            plus_score,
                            minus_score,
                        ),
                        axis=1,
                    ).astype(
                        np.float32
                    )

                    count = (
                        batch
                        .count
                        .numpy()
                        .astype(
                            np.float32
                        )
                    )

                    mask = (
                        batch
                        .mask
                        .numpy()
                        .astype(
                            bool
                        )
                    )

                    score_chunks.append(
                        score[
                            mask
                        ]
                    )

                    count_chunks.append(
                        count[
                            mask
                        ]
                    )

                    # 32-bp diagnostic bridge to EXP033.
                    n_regions = (
                        TARGET_LENGTH
                        // REGION_WIDTH
                    )

                    score_r = (
                        score
                        .reshape(
                            B,
                            2,
                            n_regions,
                            REGION_WIDTH,
                        )
                        .max(
                            axis=-1
                        )
                    )

                    count_r = (
                        count
                        .reshape(
                            B,
                            2,
                            n_regions,
                            REGION_WIDTH,
                        )
                    )

                    mask_r = (
                        mask
                        .reshape(
                            B,
                            2,
                            n_regions,
                            REGION_WIDTH,
                        )
                    )

                    valid_r = (
                        mask_r
                        .all(
                            axis=-1
                        )
                    )

                    tss_count_r = (
                        (
                            (
                                count_r
                                > 0
                            )
                            & mask_r
                        )
                        .sum(
                            axis=-1
                        )
                        .astype(
                            np.int16
                        )
                    )

                    positive_r = (
                        tss_count_r
                        > 0
                    )

                    regional_score_chunks.append(
                        score_r[
                            valid_r
                        ].astype(
                            np.float32
                        )
                    )

                    regional_positive_chunks.append(
                        positive_r[
                            valid_r
                        ].astype(
                            np.bool_
                        )
                    )

                    regional_tss_count_chunks.append(
                        tss_count_r[
                            valid_r
                        ]
                    )

                    positions += int(
                        mask.sum()
                    )

                    finished = (
                        start
                        + B
                        >= len(
                            tiles
                        )
                    )

                    if (
                        batch_no == 1
                        or batch_no
                        % progress_every
                        == 0
                        or finished
                    ):
                        elapsed = (
                            time.perf_counter()
                            - started
                        )

                        print(
                            f"tiles="
                            f"{start + B:>5}"
                            f"/{len(tiles)} "
                            f"positions="
                            f"{positions:>10,} "
                            f"positions/s="
                            f"{positions / elapsed:,.0f}"
                        )

        fallback = [
            str(
                w.message
            )
            for w in caught
            if "fall back"
            in str(
                w.message
            ).lower()
        ]

        if fallback:
            raise RuntimeError(
                "DirectML fallback: "
                + " | ".join(
                    fallback
                )
            )

    finally:
        model.train(
            was_training
        )

    scores = np.concatenate(
        score_chunks
    )

    counts = np.concatenate(
        count_chunks
    )

    positive = (
        counts
        > 0
    )

    print()
    print(
        "Computing per-base AP..."
    )

    per_base_ap = float(
        average_precision_score(
            positive,
            scores,
        )
    )

    positive_scores = (
        scores[
            positive
        ]
    )

    positive_counts = (
        counts[
            positive
        ]
    )

    print(
        "Computing EPIC dense-rank Spearman..."
    )

    pred_rank = rankdata(
        positive_scores,
        method="dense",
    )

    count_rank = rankdata(
        positive_counts,
        method="dense",
    )

    per_base_spearman = float(
        pearsonr(
            pred_rank,
            count_rank,
        )[
            0
        ]
    )

    regional_scores = np.concatenate(
        regional_score_chunks
    )

    regional_positive = np.concatenate(
        regional_positive_chunks
    )

    regional_tss_count = (
        np.concatenate(
            regional_tss_count_chunks
        )
        .astype(
            np.int64
        )
    )

    regional_ap32 = float(
        average_precision_score(
            regional_positive,
            regional_scores,
        )
    )

    order = np.argsort(
        -regional_scores,
        kind="mergesort",
    )

    total_tss = int(
        regional_tss_count.sum()
    )

    top10_n = int(
        math.ceil(
            0.10
            * len(
                order
            )
        )
    )

    top10_tss_recall = float(
        regional_tss_count[
            order[
                :top10_n
            ]
        ].sum()
        / total_tss
    )

    cumulative_tss = np.cumsum(
        regional_tss_count[
            order
        ]
    )

    needed_95 = int(
        np.searchsorted(
            cumulative_tss,
            0.95
            * total_tss,
            side="left",
        )
        + 1
    )

    fraction_regions_for_95_tss = float(
        needed_95
        / len(
            order
        )
    )

    elapsed = (
        time.perf_counter()
        - started
    )

    summary = pd.DataFrame([
        {
            "experiment": EXPERIMENT,
            "step": STEPS,
            "valid_position_strand": int(
                len(
                    scores
                )
            ),
            "positive_positions": int(
                positive.sum()
            ),
            "per_base_average_precision": (
                per_base_ap
            ),
            "epic_dense_rank_spearman": (
                per_base_spearman
            ),
            "regional_ap32_from_max_base_score": (
                regional_ap32
            ),
            "top10_region_tss_recall": (
                top10_tss_recall
            ),
            "fraction_regions_for_95pct_tss": (
                fraction_regions_for_95_tss
            ),
            "validation_minutes": float(
                elapsed
                / 60.0
            ),
            "validation_positions_per_second": float(
                positions
                / elapsed
            ),
        }
    ])

    return (
        summary,
        scores,
        counts,
    )


In [3]:
# ============================================================
# FULL HOLDOUT @ 10K
# ============================================================

CHECKPOINT_STEP = 10_000

CHECKPOINT_PATH = (
    checkpoint_path(
        CHECKPOINT_STEP
    )
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

assert (
    checkpoint[
        "experiment"
    ]
    == EXPERIMENT
)

assert (
    checkpoint[
        "architecture"
    ]
    == ARCHITECTURE
)

validation_model = (
    create_model()
    .to(device)
)

validation_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print()
print(
    "=" * 72
)

print(
    "FULL HOLDOUT @ 10K"
)

print(
    "=" * 72
)

(
    holdout_summary,
    holdout_scores,
    holdout_counts,
) = evaluate_full_holdout(
    validation_model,
    validation_tiles,
    batch_size=4,
    progress_every=250,
)

display(
    holdout_summary
)

SUMMARY_PATH = (
    RUN_DIR
    / "holdout_summary_10k.csv"
)

holdout_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

print()
print(
    "Saved:",
    SUMMARY_PATH,
)


NameError: name 'checkpoint_path' is not defined

## Как читать EXP034

Первым делом сравниваем не только одну цифру.

### 1. Настоящая EPIC-постановка

```text
per_base_average_precision
epic_dense_rank_spearman
```

Это впервые в нашей текущей серии снова **base-resolution** оценка.

### 2. Мост к EXP033

```text
regional_ap32_from_max_base_score
top10_region_tss_recall
fraction_regions_for_95pct_tss
```

Для ориентира EXP033 @10k:

```text
regional AP32        ≈ 0.1194
top10 TSS recall     ≈ 0.7170
95% TSS fraction     ≈ 0.4347
```

Не требуем, чтобы per-base модель мгновенно выиграла все regional diagnostics:
она оптимизирует уже другую и более правильную задачу.

### 3. Что будет успехом

Особенно интересно, если одновременно:

```text
per-base AP растёт относительно старых base-level моделей
+
Spearman становится осмысленным
+
regional AP32 не разваливается
```

Тогда следующий шаг — **EXP035 hard-example curriculum**:
подмешивать LOW-score TP и HARD_FP уже в обучение.


In [ ]:
# ============================================================
# FINAL CELL — SAVE ALL IMPORTANT RESULTS INTO ONE JSON
# ============================================================

RESULTS_JSON = (
    RUN_DIR
    / "EXP034_RESULTS.json"
)

history_df = pd.DataFrame(
    history
)

payload = {
    "experiment": {
        "name": EXPERIMENT,
        "architecture": ARCHITECTURE,
        "step": int(
            CHECKPOINT_STEP
        ),
        "objective": (
            "masked per-base Poisson NLL on pooled counts"
        ),
        "input_channels": 5,
        "target_length": int(
            TARGET_LENGTH
        ),
        "context_flank": int(
            CONTEXT_FLANK
        ),
        "input_length": int(
            INPUT_LENGTH
        ),
        "local_rf_bp": 35,
        "context_rf_bp": int(
            CONTEXT_RF_BP
        ),
        "positive_branch_fraction": 0.5,
        "batch_size": int(
            BATCH_SIZE
        ),
        "learning_rate": float(
            LEARNING_RATE
        ),
        "seed": int(
            SEED
        ),
    },

    "holdout_summary": (
        holdout_summary
        .where(
            pd.notnull(
                holdout_summary
            ),
            None,
        )
        .to_dict(
            orient="records"
        )
    ),

    "training": {
        "n_logged_steps": int(
            len(
                history_df
            )
        ),
        "final_loss": float(
            history_df[
                "loss"
            ].iloc[
                -1
            ]
        ),
        "final_ema_loss": float(
            history_df[
                "ema_loss"
            ].iloc[
                -1
            ]
        ),
        "final_target_positions_per_second": float(
            history_df[
                "target_positions_per_second"
            ].iloc[
                -1
            ]
        ),
        "history_every_100_steps": (
            history_df[
                (
                    history_df[
                        "step"
                    ]
                    == 1
                )
                | (
                    history_df[
                        "step"
                    ]
                    % 100
                    == 0
                )
            ]
            .where(
                pd.notnull(
                    history_df
                ),
                None,
            )
            .to_dict(
                orient="records"
            )
        ),
    },

    "comparison_reference": {
        "EXP033_step_10k_regional_ap32": (
            0.119389
        ),
        "EXP033_step_10k_top10_tss_recall": (
            0.717029
        ),
        "EXP033_step_10k_fraction_regions_for_95pct_tss": (
            0.434711
        ),
    },

    "notes": [
        (
            "EXP034 deliberately does not use hard-example mining yet; "
            "the goal is to isolate dual-path + per-base Poisson."
        ),
        (
            "Current project generator provides 5 input channels; "
            "soft-mask/lowercase is not added in this experiment."
        ),
        (
            "Regional metrics are diagnostics computed by max-pooling "
            "per-base log-rate inside each 32-bp region."
        ),
    ],
}

RESULTS_JSON.write_text(
    json.dumps(
        payload,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print()
print("=" * 72)
print("ГОТОВО")
print("=" * 72)
print()
print(
    "Пришли мне этот файл:"
)
print()
print(
    RESULTS_JSON
)
